# Indexing MS MARCO v1 Passage by OpenSearch for BM25 Model

- [msmarco-passage](https://ir-datasets.com/msmarco-passage.html)
- Prerequisite: corpus downloaded via [dataset/msmarco-v1-passage](../../dataset/msmarco-v1-passage/README.md)

In [ ]:
import sys
!{sys.executable} -m pip install -q ir_datasets pandas opensearch-py dotenv

In [ ]:
import pprint
from tqdm import tqdm

### Create an OpenSearch Client

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [ ]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

### Index a Corpus for BM25 Model

In [ ]:
import ir_datasets
dataset_name = "msmarco-passage"
dataset = ir_datasets.load(dataset_name)

In [ ]:
index_name = "msmarco_v1_passage_bm25"

In [ ]:
# Delete an existing index (be careful)
if client.indices.exists(index=index_name):
    response = client.indices.delete(index=index_name)
    pprint.pprint(response)
else:
    print(f"{index_name} does not exist")

In [ ]:
index_body = {
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 0
    }
    # English corpus: rely on OpenSearch's default (standard) analyzer.
  },
  "mappings": {
    "properties": {
        "docid": { "type": "keyword" },
        "text": { "type": "text" },
    }
  }
}

response = client.indices.create(index=index_name, body=index_body)
pprint.pprint(response)

Indexing (8.8M passages, no server-side pipeline — expect this to be I/O bound; roughly 30-60 min)

In [ ]:
def prepare_documents(dataset):
    """
    Yield raw bulk actions. MS MARCO passages are GenericDoc (doc_id + text
    only, no title) and already passage-sized (~56 words on average), so
    unlike the Robust04 notebooks there is no chunking stage and no
    MAX_DOC_CHARS cap.
    """
    for doc in dataset.docs_iter():
        yield {
            "_id": doc.doc_id,  # Unique identifier for the passage
            "_source": {
                "docid": doc.doc_id,
                "text": doc.text,
            }
        }

In [ ]:
from opensearchpy.helpers import streaming_bulk

# Total for the progress bar (docs_count is instant for msmarco-passage).
total = dataset.docs_count()   # 8,841,823

success, errors = 0, []
with tqdm(total=total, desc="Indexing") as bar:
    for ok, item in streaming_bulk(
        client,
        prepare_documents(dataset),
        index=index_name,
        chunk_size=1000,
        request_timeout=300,
        max_retries=3,
        initial_backoff=2,
        raise_on_error=False,          # collect failures instead of aborting the run
        raise_on_exception=False,
    ):
        bar.update(1)                  # advances per actually-processed passage
        success += ok
        if not ok:
            errors.append(item)

print(f"indexed: {success},  failed: {len(errors)}")
if errors:
    pprint.pprint(errors[:3])          # inspect the first few errors

---
### (Optional) Re-index passages missing from the first pass

BM25 indexing has no server-side pipeline, so this is just a completeness
check: diff the corpus against what's in the index and re-index anything
missing.

In [ ]:
from opensearchpy.helpers import scan

# All ids actually in the index (_source disabled -> fast).
# 8.8M ids fit comfortably in memory (a few hundred MB), but the scan takes a while.
indexed = set()
for hit in scan(
    client,
    index=index_name,
    query={"query": {"match_all": {}}, "_source": False},
    size=5000,
):
    indexed.add(hit["_id"])

# All ids the dataset should have produced
all_ids = {doc.doc_id for doc in dataset.docs_iter()}

missing = sorted(all_ids - indexed)
print(f"indexed: {len(indexed)},  missing: {len(missing)}")
print(missing[:10])

In [ ]:
# Re-index the passages identified as missing by the scan diff, printing every error.
from opensearchpy.helpers import streaming_bulk

def prepare_missing(dataset, missing_ids):
    docstore = dataset.docs_store()
    for doc_id in missing_ids:
        doc = docstore.get(doc_id)
        yield {
            "_id": doc_id,
            "_source": {"docid": doc_id, "text": doc.text},
        }

retry_ok, retry_failed = 0, []
with tqdm(total=len(missing), desc="Re-indexing") as bar:
    for ok, item in streaming_bulk(
        client,
        prepare_missing(dataset, missing),
        index=index_name,
        chunk_size=500,
        request_timeout=300,
        raise_on_error=False,
        raise_on_exception=False,
    ):
        bar.update(1)
        retry_ok += ok
        if not ok:
            retry_failed.append(item)

print(f"retried ok: {retry_ok}, still failing: {len(retry_failed)}\n")

for item in retry_failed[:10]:     # full error detail for the first failures
    pprint.pprint(item)
    print("-" * 80)